In [18]:
from dotenv import load_dotenv
import os
from pathlib import Path

In [3]:
print(os.getenv("OPENAI_API_KEY"))

your_local_openai_api_key_here


In [19]:
global_env_path = Path.home() / ".env"
print(global_env_path)

/Users/karanchanana/.env


In [ ]:
global_env_path = Path.home() / ".env"
api_key = os.getenv("GOOGLE_API_KEY")
print(api_key)

None


In [5]:
#!pip install -U google-generativeai

In [6]:
#!pip install langchain google-generativeai langchain-google-genai langchain_community chromadb pandas

In [6]:
tavily_api_key = os.getenv("TAVILY_API_KEY")
print(tavily_api_key)

None


In [20]:
load_dotenv(global_env_path)
api_key = os.getenv("GOOGLE_API_KEY")
#print(api_key)
tavily_api_key = os.getenv("TAVILY_API_KEY")
#print(tavily_api_key)

In [9]:
#pip install ipywidgets

In [21]:
from langchain.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.docstore.document import Document

In [22]:
from langchain.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_google_genai import ChatGoogleGenerativeAI

# **Build Your First AI Agent**

### 🤖 **Rule based Agent**

First we'll build a **simple rule-based AI agent** — a chatbot that responds to user inputs based on predefined rules.

This is your first step toward understanding how AI agents work. Unlike machine learning models, rule-based agents follow **if-then** logic to make decisions.

By the end of this part, you will:
- Understand what a rule-based system is
- Build a chatbot that can respond to basic inputs
- See where rule-based agents fall short

#### 🧩 **What is a Rule-Based Agent?**

A **rule-based agent** operates on hard-coded rules. These rules define what the system should do when it encounters specific input.

For example:

- If someone says "hello", respond with "Hi there!"
- If someone asks for your name, respond with "I'm a chatbot!"

This type of system does **not learn** or adapt — it's predictable and easy to implement, but not flexible.

You'll now create your first rule-based chatbot below.


In [11]:
# 🔧 Rule-based chatbot function

def rule_based_chatbot(user_input):
    user_input = user_input.lower()

    if "hello" in user_input or "hi" in user_input:
        return "Hello! How can I assist you today?"
    elif "your name" in user_input:
        return "I'm a simple rule-based chatbot."
    elif "weather" in user_input:
        return "I can't provide live weather updates, but it's a nice day to code!"
    elif "bye" in user_input or "goodbye" in user_input:
        return "Goodbye! Have a great day."
    else:
        return "I'm not sure how to respond to that. Can you try rephrasing?"

#####💡 **Try it!**

Type something like:
- "Hi"
- "What's your name?"
- "Tell me the weather"
- "Bye"

To exit the chatbot, type `exit`.

Think about:
- What happens when you ask something unexpected?
- Can you add more rules to improve it?


In [12]:
# while True:
#     user_input = input("You: ")
#     if user_input.lower() == "exit":
#         print("Chatbot: Session ended.")
#         break
#     response = rule_based_chatbot(user_input)
#     print("Chatbot:", response)

### 🧠 **Simple LLM Powered Chatbot**

In this part, you'll create a chatbot using **Google's Gemini LLM**.

It will:
- Take user input
- Use `gemini-2.0-flash
` model to generate responses
- Act as a simple conversational agent

Let's start!


In [13]:
import google.generativeai as genai

In [15]:
#print(api_key)

In [14]:
genai.configure(api_key=api_key)

In [15]:
# Load the Gemini Pro model for chat
model = genai.GenerativeModel("gemini-2.0-flash")

# Start a chat session (enables memory across turns)
chat = model.start_chat()

In [16]:
def chat_with_gemini(user_input):
    response = chat.send_message(user_input)
    return response.text

In [17]:
print("🤖 Gemini Chatbot Ready! Type 'exit' to quit.\n")

# while True:
#     user_input = input("You: ")
#     if user_input.lower() in ["exit", "quit"]:
#         print("Bot: Goodbye!")
#         break
#     reply = chat_with_gemini(user_input)
#     print("Bot:", reply)

🤖 Gemini Chatbot Ready! Type 'exit' to quit.



### 🩺 **Medical RAG-Based AI Agent with Web Search**

#####📌 **Business Problem**
In today's digital healthcare landscape, patients and medical professionals frequently seek accurate, up-to-date answers to a wide range of medical questions. However:

- Search engines return too much noise or unverified sources.

- Chatbots often hallucinate or give generic advice without referencing valid information.

- Medical professionals/Patients have limited time to find contextually relevant evidence or literature.

There is a growing need for an AI system that can:

- Provide precise, evidence-backed answers,

- Maintain context awareness during multi-turn conversations,

- And optionally enhance its knowledge using trusted real-time web sources.

##### 🎯 **Objectives**
This project aims to build a Retrieval-Augmented Generation (RAG)-based Medical Assistant chatbot that:

✅ Answers healthcare-related questions accurately using an internal medical dataset

✅ Retrieves relevant information from a ChromaDB vector store

✅ Optionally fetches real-time results via the Tavily web search API

✅ Maintains conversation context and memory using LangChain

##### **1. Setup and Prerequisites**
    
Before we start building, make sure you have all the necessary libraries and tools set up.

Required Libraries:

- Langchain: For building the RAG system and interacting with models.

- Google Generative AI: For embedding generation and leveraging LLMs.

- Tavily API: For web search capabilities.

- Chroma: For storing and querying embeddings efficiently.
    
Run this cell to install dependencies.

In [23]:
import pandas as pd

df = pd.read_csv(r"ai-medical-chatbot.csv")

In [20]:
df.head()

,Description,Patient,Doctor
0,Q. What does abutment of the nerve root mean?,"Hi doctor,I am just wondering what is abutting...",Hi. I have gone through your query with dilige...
1,Q. What should I do to reduce my weight gained...,"Hi doctor, I am a 22-year-old female who was d...",Hi. You have really done well with the hypothy...
2,Q. I have started to get lots of acne on my fa...,Hi doctor! I used to have clear skin but since...,Hi there Acne has multifactorial etiology. Onl...
3,Q. Why do I have uncomfortable feeling between...,"Hello doctor,I am having an uncomfortable feel...",Hello. The popping and discomfort what you fel...
4,Q. My symptoms after intercourse threatns me e...,"Hello doctor,Before two years had sex with a c...",Hello. The HIV test uses a finger prick blood ...


In [21]:
df.count()

Description    256916
Patient        256916
Doctor         256916
dtype: int64

###### 🧾 **Dataset Description**

The dataset used in this project consists of real-world medical Q&A data sourced from an online medical consultation platform. It simulates conversations between patients and doctors.

📂 **Dataset Columns:**
- **Description**: A brief summary of the medical query (useful for indexing or quick reference).
- **Patient**: The full question asked by the patient, often containing symptoms, background, and concerns.
- **Doctor**: The detailed response provided by a medical professional, including explanations, possible diagnoses, treatments, and referrals.

📊 **Dataset Characteristics:**
- 🔢 ~256916 records
- 📚 Topics include: dermatology, gynecology, psychiatry, endocrinology, urology, gastroenterology, and more
- 📌 Mostly **conversational** and **medical-context-rich**
- 🧠 Great for building **retrieval-augmented** LLM systems where factual and domain-specific grounding is essential

🎯** Why This Dataset?**
- Provides real, nuanced **medical consultation dialogue**
- Questions are **informal and varied**, closely resembling real user queries
- Responses offer structured medical **advice, terminology, and reasoning**
- Helps improve the **grounding** of LLM responses using vector search (via ChromaDB)

This dataset is embedded into a **vector store** using `GoogleGenerativeAIEmbeddings` and retrieved during RAG pipeline execution to enhance factual accuracy and relevance in LLM-generated answers.


In [22]:
df.describe()

,Description,Patient,Doctor
count,256916,256916,256916
unique,228722,246006,242150
top,Q. Please explain the poisonous effect of phen...,"Hello doctor, My fiancee and I had unprotected...",Hi. For further doubts consult a sexologist on...
freq,1137,1137,1519


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 256916 entries, 0 to 256915
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Description  256916 non-null  object
 1   Patient      256916 non-null  object
 2   Doctor       256916 non-null  object
dtypes: object(3)
memory usage: 5.9+ MB


In [23]:
df.value_counts()

Description                                                                    Patient                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      Doctor                                                                                                                                                                                                                                                                                                                      

In [24]:
df['document'] = (
    df['Description'].fillna('') + '\n\n' +
    'Patient: ' + df['Patient'].fillna('') + '\n\n' +
    'Doctor: ' + df['Doctor'].fillna('')
)

##### **4. Vector Store Initialization**
We now prepare a Chroma vector store to store and search the medical documents efficiently.

This is just a database where all the data is stored in a machine understandable format.

### Steps:
- Create Embeddings: Use Google’s Generative AI embeddings to convert documents into vector representations.

- Initialize Chroma DB: Store the document vectors for fast similarity search.

- Set Up Retriever: This is used to fetch relevant documents based on user queries.

#### ***Explanation***
- Embeddings: These are vector representations of text. Each document is converted into a numerical vector that represents its meaning.

- Vector Store: Chroma stores these vectors and enables efficient similarity searches.

In [26]:
import shutil

In [27]:
PERSIST_DIR = "chroma_db"
if os.path.exists(PERSIST_DIR):
   shutil.rmtree(PERSIST_DIR)

In [9]:
from langchain.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.docstore.document import Document

In [10]:
# Initialize the embedding model
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

In [11]:

# Set up the vector store
PERSIST_DIR = "chroma_db"
#os.path.exists(PERSIST_DIR) and os.listdir(PERSIST_DIR):
if os.path.exists(os.path.join(PERSIST_DIR, "chroma.sqlite3")):
    # Load existing DB if it exists
    vectorstore = Chroma(persist_directory=PERSIST_DIR, embedding_function=embeddings)
else:
    # Create documents from the dataframe
    documents = [Document(page_content=row['document']) for _, row in df.head(1000).iterrows()]
    # Create and persist new DB
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory=PERSIST_DIR
    )
    vectorstore.persist()

# Set up retriever
retriever = vectorstore.as_retriever()

/var/folders/hc/_nzb4lgs3vl7f22df0r_pj240000gn/T/ipykernel_16113/4087796037.py:16: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


Trying differnt Embedding model as gemini is giving 504 error

In [29]:
#pip install -U sentence-transformers


In [30]:
from sentence_transformers import SentenceTransformer


In [31]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [33]:
from langchain_community.embeddings import SentenceTransformerEmbeddings



In [34]:
# Choose a suitable SentenceTransformer model
model_name = "sentence-transformers/all-MiniLM-L6-v2" # Example model
embeddings = SentenceTransformerEmbeddings(model_name=model_name)

/var/folders/hc/_nzb4lgs3vl7f22df0r_pj240000gn/T/ipykernel_16113/514000237.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name=model_name)


In [32]:
embeddings = model.encode(df['document']).tolist() # Convert to list for ChromaDB

KeyboardInterrupt: 

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader # Or other loaders like PDFMiner, UnstructuredFileLoader, etc.


# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

In [40]:

# Create the Chroma vector store from the chunks and embeddings
# persist_directory is optional for local persistence

PERSIST_DIR = "./chroma_db"

if os.path.exists(os.path.join(PERSIST_DIR, "chroma.sqlite3")):
    # Load existing DB if it exists
    vectorstore = Chroma(persist_directory=PERSIST_DIR, embedding_function=embeddings)
else:
    # Create documents from the dataframe
    documents = [Document(page_content=row['document']) for _, row in df.iterrows()]
      # Create and persist new DB
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory=PERSIST_DIR
    )
vectorstore.persist()


In [41]:

# Set up retriever
retriever = vectorstore.as_retriever()

##### **5. Building the Retrieval-Augmented Generation (RAG) Chain**
This part builds the core functionality of our chatbot using a RAG system. It retrieves relevant documents based on user queries and then uses a language model (LLM) to generate answers based on the retrieved context.

***Explanation***
- The ```system prompt``` tells the LLM how to use the retrieved context.

- ```create_stuff_documents_chain``` combines the retrieved documents into the input for the LLM.

- ```create_retrieval_chain``` connects the retriever (which fetches relevant documents) and the chain (which processes them).

**TEST RUN**

In [12]:
from langchain.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_google_genai import ChatGoogleGenerativeAI

# Define the system prompt that will guide the chatbot
system_prompt = (
    "You are a medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise. If the question is not clear, ask follow-up questions."
    "\n\n"
    "{context}"
)

# Create a chat prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# Initialize the LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

# Create the RAG chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)


##### **6. Testing the RAG System**
Now, let’s test the RAG system by asking a sample medical question. The chatbot will retrieve relevant documents and generate an answer based on those documents.

###### ***Explanation***
```Invoke```: This triggers the retrieval and generation process. The input is the user query.

In [13]:
# Example usage of the RAG chain
response = rag_chain.invoke({"input": "Can you tell me about the symptoms of diabetes?"})
print("Response is\n------")
print(response["answer"])

Response is
------
I apologize, but the provided context does not explicitly list the general symptoms of diabetes. It discusses complications like erectile dysfunction and nerve damage, as well as the management of existing diabetes.


In [14]:
# Example usage of the RAG chain
response = rag_chain.invoke({"input": "Why do i get frustrated?"})
print("Response is\n------")
print(response["answer"])

Response is
------
You get easily frustrated and have a low tolerance for anger, getting mad over various things. These symptoms, along with difficulty focusing, hyperactivity, and impulsivity, are suggestive of Attention-Deficit Hyperactivity Disorder (ADHD). ADHD runs in your family, which further supports this possibility.


In [ ]:
# Example usage of the RAG chain
response = rag_chain.invoke({"input": "What is ALS?"})
print("Response is\n------")
print(response["answer"])

Response is
------
I don't know what ALS is based on the provided information. The context mentions ALS (amyotrophic lateral sclerosis) in relation to muscle twitches and the absence of weakness, but it does not provide a definition of the condition itself.


In [16]:
# Example usage of the RAG chain
response = rag_chain.invoke({"input": "What type of cancer is a germ cell tumor?"})
print("Response is\n------")
print(response["answer"])

Response is
------
I don't know the answer to this question based on the provided information.


In [17]:
# Example usage of the RAG chain
response = rag_chain.invoke({"input": "I am having swollen lymph node in my neck. Will it be painful after adjusting my neck?"})
print("Response is\n------")
print(response["answer"])

Response is
------
Based on the provided information, the doctor does not think a chiropractor's neck adjustment will cause any trouble to a lymph node. The pain you are experiencing might be related to an intervertebral disc injury or nerve root entrapment, which can cause muscle soreness and radiating symptoms. An MRI scan of the cervical spine and hot water fermentation are suggested for these issues.


In [28]:
# Example usage of the RAG chain
response = rag_chain.invoke({"input": "Can you tell me about the top tourist places in Japan?"})
print("Response is\n------")
print(response["answer"])

Response is
------
I am sorry, but I cannot provide information about the top tourist places in Japan. My knowledge base is limited to medical questions based on the provided context, and it does not contain information on travel or tourism.



##### ***7. History Aware RAG System***

Enhance the RAG chain to consider conversation history. This allows the assistant to understand follow-up questions that refer to previous turns in the conversation.

*   **Contextualization Prompt:** Define a system prompt specifically for rephrasing a user's follow-up question into a standalone question, using the chat history for context. The `MessagesPlaceholder("chat_history")` will hold the conversation history.
*   **History-Aware Retriever:** Use `create_history_aware_retriever` to create a retriever that first uses the LLM and the contextualization prompt to potentially rephrase the user's input based on history, *before* querying the vector store.

###### ***Explanation***
`History-Aware Retriever`: This component understands previous queries and context, enabling the assistant to handle follow-up questions more intelligently.

`MessagesPlaceholder`: This stores and passes the chat history.

#### ***7.1. Testing the History Aware RAG Chain***

Test the history-aware chain with a sequence of questions:
1.  Ask an initial question ("Breast Cancer Screening?").
2.  Store the question and the AI's response in the `chat_history`.
3.  Ask a follow-up question ("what screenings should be done?").
4.  Invoke the chain again with the new question and the updated `chat_history`.
Observe how the chain handles the follow-up question using the context from the history.

In [18]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain.chains import ConversationalRetrievalChain

# Assuming the `rag_chain` is created correctly
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

chat_history = []

In [30]:
# First question
question = "what is optical nerve damage"
ai_msg_1 = rag_chain.invoke({"question": question, "chat_history": chat_history})

chat_history.append((question, ai_msg_1["answer"]))  # Use ai_msg_1 for storing the answer

# Print the result of the first question
print(ai_msg_1["answer"])

Optical nerve damage refers to a gradual and irreversible deterioration of the optic nerve. This damage can be significant, even "near total," as indicated by measures like CDR 0.9. The doctor explains that the optic nerve is like the "film" of a camera; if it's compromised, changing the "lens" (like a cataract) won't fully restore vision. It leads to vision problems such as low and blurry vision.


In [31]:
# Second question
second_question = "What is the treatment for it?"
ai_msg_2 = rag_chain.invoke({"question": second_question, "chat_history": chat_history})

# Print the result of the second question
print(ai_msg_2["answer"])

# You can also print the chat history to see the conversation so far
#print(chat_history)

Based on the provided context:

*   The changes in the optic nerve are generally **irreversible**, and vision loss that has already occurred due to optic nerve damage is **not reversible**.
*   The goal of treatment is to **prevent further damage** and preserve whatever optic nerve function is left.
*   **Brimonidine eye drops** have been mentioned as potentially helpful for neuroprotection and preserving remaining optic nerve function, theoretically.
*   In cases like glaucoma, the goal is to **lower the pressure** to prevent further damage to the optic nerve.
*   Cataract surgery, while addressing a lens issue, may not improve vision if the optic nerve (retina) is significantly compromised, and can be risky depending on the patient's overall health.


##### **8. Conversational Agent with Web Search (Tavily)**

Implement a conversational agent that uses the Tavily search engine for questions that cannot be answered from the local RAG context and maintains conversation history.

*   **Import Tavily Tool:** Import the `TavilySearchResults` tool.
*   **Initialize Tavily Tool:** Create an instance of the Tavily search tool.

In [19]:
from langchain_community.tools.tavily_search import TavilySearchResults

# Initialize the Tavily tool
tavily_tool = TavilySearchResults()

/var/folders/hc/_nzb4lgs3vl7f22df0r_pj240000gn/T/ipykernel_85063/3608840554.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults()


In [33]:
search_results = tavily_tool.run("chatgpt")
print(search_results[0]['content'])

Barbara Pazur

8 min read

ChatGPT is the artificial intelligence chatbot that started the whole generative AI revolution. It's a multimodal tool that can perform a range of functions, from answering complex questions -- and simplifying the answers for you, as if you're 5 years old -- to coding software, creating images and videos and even engaging in casual conversations. [...] ChatGPT is built on a transformer architecture, specifically the GPT (generative pretrained transformer) family of models, ergo the name ChatGPT. It was trained on massive amounts of data from books and the internet -- websites, Wikipedia, Reddit threads, news sites and much, much more.

(Disclosure: Ziff Davis, CNET's parent company, in April filed a lawsuit against OpenAI, alleging it infringed Ziff Davis copyrights in training and operating its AI systems.) [...] Launched in November 2022 by OpenAI, the company co-founded by Sam Altman, ChatGPT quickly became the most popular AI application ever released, an

*   **Initialize Conversation Memory:** Set up `ConversationBufferMemory` to store the chat history. `return_messages=True` ensures history is stored as message objects.

In [20]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


/var/folders/hc/_nzb4lgs3vl7f22df0r_pj240000gn/T/ipykernel_85063/4063055825.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


*   **Initialize Conversational Agent:**
    *   Use `initialize_agent` with the `CONVERSATIONAL_REACT_DESCRIPTION` agent type, which is designed for conversation and uses a ReAct (Reasoning and Acting) framework.
    *   Provide the LLM (`gemini-1.5-flash`), the Tavily tool, and the conversation memory.

In [21]:
from langchain.agents import initialize_agent, AgentType
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.2) #try different models here
tools = [ tavily_tool ]
agent = initialize_agent(
    tools=tools,
    llm = llm,
    memory=memory,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION
)

/var/folders/hc/_nzb4lgs3vl7f22df0r_pj240000gn/T/ipykernel_85063/803951214.py:4: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


*   **Define Message Handler:** Create a simple function `handle_user_message` to run the agent with user input.

In [22]:
def handle_user_message(user_input):
    response = agent.run(user_input)
    return response


##### **9.1. Testing the Conversational Agent**

Test the conversational agent with various inputs:
*   A question requiring web search ("Who won yesterday's ipl match?").
*   An instruction modifying the agent's behavior ("Don't talk about anything other than stomach issues.").
*   A question violating the instruction ("What is bird flu?").
This demonstrates the agent's ability to use tools (Tavily) and adhere to conversational context stored in memory.

In [37]:
handle_user_message("What happened yeserday between India & Pakistan?")

/var/folders/hc/_nzb4lgs3vl7f22df0r_pj240000gn/T/ipykernel_65992/2106510748.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = agent.run(user_input)


'Based on my search results, there was a significant escalation of tensions between India and Pakistan around late April and early May 2025.  This involved a deadly attack on Indian tourists in Kashmir, which India blamed on Pakistan.  India responded with several actions, including suspending the Indus Waters Treaty and imposing travel restrictions.  Pakistan denied involvement and also took retaliatory measures.  The situation was described as the most serious point in bilateral relations in recent years.  A ceasefire eventually occurred, but the details surrounding its negotiation and the exact events of each day are not fully detailed in the readily available information.  To get a precise day-by-day account of what happened yesterday specifically, you would need to specify the exact date and consult more detailed news archives from that time.\n```'

In [38]:
handle_user_message("Can you tell me about the top tourist places in Japan?")

'Japan offers a diverse range of tourist attractions, catering to various interests.  To give you the "top" places is subjective and depends on your preferences, but here are some consistently popular choices, categorized for easier planning:\n\n**Nature & Scenery:**\n\n* **Mount Fuji:** Iconic volcano, offering stunning views and hiking opportunities (seasonal).\n* **Japanese Alps:** Breathtaking mountain ranges perfect for hiking, skiing, and enjoying onsen (hot springs).\n* **Kyoto\'s Arashiyama Bamboo Grove:** A magical, serene path through towering bamboo stalks.\n* **Miyajima Island\'s Itsukushima Shrine:** Famous for its "floating" torii gate.\n* **Hakone:** Volcanic hot springs resort town with views of Mount Fuji and stunning lake scenery.\n\n**Culture & History:**\n\n* **Kyoto:** Ancient capital, rich in temples, shrines, geishas, traditional gardens, and palaces (e.g., Kinkaku-ji - Golden Pavilion, Fushimi Inari Shrine).\n* **Tokyo:** Modern metropolis with a blend of tradit

In [39]:
handle_user_message("Don't talk about anything other than stomach issues.")

"I understand.  Let's talk only about stomach issues. To best assist you, could you please tell me more about what you'd like to know?  For example, are you experiencing a specific problem, are you looking for general information about digestive health, or are you interested in a particular type of stomach issue?  The more details you can provide, the better I can help.\n```"

In [40]:
handle_user_message("What is bird flu?")

"Bird flu, or avian influenza, refers to a group of highly contagious viral infections affecting birds.  These viruses are classified into subtypes based on their surface proteins, hemagglutinin (H) and neuraminidase (N).  While most subtypes only affect birds, some, like H5N1 and H7N9, have caused outbreaks in humans, although human-to-human transmission is rare.\n\nSymptoms in birds can range from mild respiratory illness to severe disease, often leading to death.  In humans, symptoms can resemble the flu, including fever, cough, sore throat, muscle aches, and sometimes pneumonia.  However, severe cases can be fatal.\n\nThe spread of bird flu is primarily through contact with infected birds or contaminated surfaces.  Outbreaks often lead to culling of poultry to prevent further spread.  Public health measures, such as hand hygiene and avoiding contact with sick or dead birds, are crucial in preventing human infection.  There are vaccines available for poultry, and antiviral medicatio

In [41]:
handle_user_message("what us the gemini-2.5-flash model api limit?")

"The Gemini-2.5-flash model API limits vary depending on the specific API and your access level (free tier vs. paid).  Information from various sources indicates the following:\n\n* **Token Limits:**  A common limit is 1,048,576 input tokens and varying output token limits (e.g., 65,536 for some text-only outputs, lower for audio/video).  The exact output token limit depends on the specific API endpoint used.\n\n* **Rate Limits:** Rate limits are expressed in requests per minute (RPM), requests per hour (RPH), and requests per day (RPD).  These limits also vary significantly depending on the API endpoint and access level.  Some sources mention limits in the thousands of requests per minute for certain endpoints, while others show much lower limits, especially on free tiers.  There are also limits on the total number of tokens enqueued in batches.\n\n* **File Size Limits:** For video input, a common limit is 2GB per file when using the recommended Google AI Files API.  Smaller limits ap

In [42]:
handle_user_message("Don't talk about anything other than cricket.")

"Okay, I understand.  Let's talk only about cricket. To best assist you, could you please tell me more about what you'd like to know?  For example, are you interested in a specific team, player, match, or aspect of the game? Are you looking for news, statistics, historical information, or something else?  The more details you provide, the better I can help.\n```"

In [43]:
handle_user_message("what us the gemini-2.5-flash model api limit?")

'As I mentioned before, the Gemini-2.5-flash model API limits are not publicly and consistently documented in a single place.  They vary based on your access level (free tier vs. paid) and the specific API endpoint used.  Information found online is often inconsistent and may not be entirely up-to-date.  To find the most accurate limits for your situation, you must consult the official Google AI Gemini API documentation.  Look for sections on rate limits, token limits (input and output), and any file size restrictions that apply to your intended use case.\n```'

**(Utility) Display Chat History**

Define and call a function `show_chat_history` to print the messages stored in the agent's memory, showing the conversation flow.

In [44]:
def show_chat_history():
    for message in memory.chat_memory.messages:
        role = "You" if message.type == "human" else "Bot"
        print(f"{role}: {message.content}")
show_chat_history()

You: What happened yeserday between India & Pakistan?
Bot: Based on my search results, there was a significant escalation of tensions between India and Pakistan around late April and early May 2025.  This involved a deadly attack on Indian tourists in Kashmir, which India blamed on Pakistan.  India responded with several actions, including suspending the Indus Waters Treaty and imposing travel restrictions.  Pakistan denied involvement and also took retaliatory measures.  The situation was described as the most serious point in bilateral relations in recent years.  A ceasefire eventually occurred, but the details surrounding its negotiation and the exact events of each day are not fully detailed in the readily available information.  To get a precise day-by-day account of what happened yesterday specifically, you would need to specify the exact date and consult more detailed news archives from that time.
```
You: Can you tell me about the top tourist places in Japan?
Bot: Japan offers 

In [49]:
#memory.chat_memory.clear()

In [50]:
def show_chat_history():
    for message in memory.chat_memory.messages:
        role = "You" if message.type == "human" else "Bot"
        print(f"{role}: {message.content}")
show_chat_history()

In [51]:
handle_user_message("Don't talk about anything other than cricket.")

"Right then! Let's talk cricket. What's on your mind?  Are you interested in a particular team, player, match, or aspect of the game?\n```"

In [52]:
handle_user_message("what us the gemini-2.5-flash model api limit?")

'The Gemini 2.5 Flash model API limits vary depending on the type of access (e.g.,  Live API vs. general API) and whether you are using a free tier or a paid tier.  The information I found is inconsistent across different sources.  To get the most accurate and up-to-date information, I recommend checking the official Google AI documentation for the Gemini API.  That documentation will provide the definitive answer on rate limits (requests per minute, requests per hour, etc.) and token limits.'

In [53]:
def show_chat_history():
    for message in memory.chat_memory.messages:
        role = "You" if message.type == "human" else "Bot"
        print(f"{role}: {message.content}")
show_chat_history()

You: Don't talk about anything other than cricket.
Bot: Right then! Let's talk cricket. What's on your mind?  Are you interested in a particular team, player, match, or aspect of the game?
```
You: what us the gemini-2.5-flash model api limit?
Bot: The Gemini 2.5 Flash model API limits vary depending on the type of access (e.g.,  Live API vs. general API) and whether you are using a free tier or a paid tier.  The information I found is inconsistent across different sources.  To get the most accurate and up-to-date information, I recommend checking the official Google AI documentation for the Gemini API.  That documentation will provide the definitive answer on rate limits (requests per minute, requests per hour, etc.) and token limits.


In [54]:
handle_user_message("cricket")

"Okay, let's talk cricket!  To give you the best response, I need a little more direction. What about cricket are you interested in? For example, are you curious about:\n\n* **Specific teams?** (e.g., Australia, India, England)  Perhaps you want to know their current standings, recent performance, or famous players.\n* **Particular players?** (e.g., Virat Kohli, Sachin Tendulkar, Ben Stokes)  Maybe you'd like to know their statistics, records, or playing style.\n* **Recent matches or tournaments?** (e.g., the Ashes, the World Cup, IPL)  We could discuss results, highlights, or key moments.\n* **Aspects of the game?** (e.g., batting techniques, bowling strategies, rules of the game)  I can provide explanations and insights.\n\nTell me what you'd like to discuss, and let's get into it!\n```"

In [55]:
handle_user_message("India")

"The Indian cricket team's current status is a bit complex and depends on what you're interested in.  My search revealed information about their current squad (though the exact players vary depending on the format and match), recent and upcoming matches, and news articles about the team.  To give you a better answer, could you be more specific?  For example, are you interested in:\n\n* **Their current ranking in international cricket?** (Test, ODI, T20)\n* **Their recent performance in a specific tournament or series?** (e.g., World Cup, IPL)\n* **News about specific players in the team?**\n* **Their upcoming fixtures?**\n\nThe more detail you provide, the more precise and helpful my response will be.\n```"

In [57]:
memory.clear()

##### **10. Hybrid RAG + Web Search Synthesis with ChatManager**

This section defines and implements the final `ChatManager` class, which orchestrates the conversation by combining internal knowledge (RAG) with external web search results (Tavily) and synthesizing them into a single response using the LLM.

**Key Features:**

*   **Synthesis Approach:** Instead of fallback, it fetches context from *both* RAG and Tavily for every relevant query.
*   **LLM Synthesizer:** Uses a dedicated prompt (`synthesis_prompt_template`) to instruct the LLM to combine the RAG and Tavily contexts logically, citing conflicts if necessary, based *only* on the provided information.
*   **Robust Context Fetching:** Includes helper methods (`_get_rag_context`, `_get_tavily_context`) to retrieve and format context from the RAG chain and Tavily Search (using `.invoke()` for better structured results).
*   **Conversation Memory:** Uses `ConversationBufferMemory` to maintain chat history, allowing context-aware conversations.
*   **Topic Blacklisting:** Allows the user to dynamically block topics using phrases like "don't talk about X". The manager checks both user input and the final generated answer against the blacklist.
*   **Error Handling:** Includes basic error handling during context retrieval and LLM invocation.

The core logic resides in the `handle_user_message` method, which orchestrates these steps for each user turn.

In [23]:
from langchain.memory import ConversationBufferMemory
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

class ChatManager:
    def __init__(self, llm, tavily_tool, retriever, rag_chain):
        self.llm = llm
        self.tavily_tool = tavily_tool
        self.retriever = retriever
        self.rag_chain = rag_chain

        self.memory = ConversationBufferMemory(
            memory_key="chat_history",
            return_messages=True,
            output_key='answer'
        )

        self.synthesis_prompt_template = ChatPromptTemplate.from_messages([
            ("system",
             "You are a helpful medical assistant. Synthesize an answer based only on the internal (RAG) and web (Tavily) context. Do not answer it on your own.\n"
             "Also just also do this:\n"
             "Rag: Yes or No\n"
             "Tavily: Yes or No\n"
             "If non-medical, say you're not allowed to discuss it."),
            MessagesPlaceholder("chat_history"),
            ("human", "{input}")
        ])

    def _get_rag_context(self, user_input, chat_history):
        try:
            rag_response = self.rag_chain.invoke({"input": user_input, "chat_history": chat_history})
            return rag_response.get("answer", "Sorry, no RAG answer found.")
        except Exception as e:
            return f"Error retrieving RAG context: {e}"

    def _get_tavily_context(self, user_input):
        try:
            search_results = self.tavily_tool.invoke({"query": user_input})
            contents = [res['content'] for res in search_results if 'content' in res]
            return "\n\n".join(contents) if contents else "No relevant web search results."
        except Exception as e:
            return f"Error retrieving Tavily context: {e}"

    def handle_user_message(self, user_input):
        chat_history = self.memory.chat_memory.messages
        rag_context = self._get_rag_context(user_input, chat_history)
        tavily_context = self._get_tavily_context(user_input)

        input_for_synthesis = {
            "input": user_input,
            "chat_history": chat_history,
            "rag_context": rag_context,
            "tavily_context": tavily_context
        }

        synthesis_chain = self.synthesis_prompt_template | self.llm
        response = synthesis_chain.invoke(input_for_synthesis)
        final_answer = getattr(response, 'content', response)

        self.memory.save_context({"input": user_input}, {"answer": final_answer})

        return final_answer.strip()

    def clear_history(self):
        self.memory.clear()

    def get_history(self):
        return self.memory.chat_memory.messages

# def rag_answer(query: str, chat_manager: ChatManager) -> str:
#     chat_history = chat_manager.memory.chat_memory.messages

#     response = chat_manager.rag_chain.invoke({
#         "input": query,
#         "chat_history": chat_history
#     })

#     answer = response.get("answer", "No answer found.")
#     chat_manager.memory.save_context({"input": query}, {"answer": answer})

#     return answer


**10.1. Interactive Testing of the Synthesizing ChatManager**

This cell runs an interactive chat loop to test the `ChatManager`.

*   It initializes an instance of the `ChatManager` class, providing the LLM, Tavily tool, RAG retriever, and RAG chain.
*   It prompts the user for input in a loop (`You: `).
*   Each input is processed by `chat.handle_user_message()`.
*   The method gathers RAG and Tavily context, calls the LLM for synthesis, checks blacklists, updates memory, and returns the final synthesized answer.
*   The bot's response is printed (`Bot: `).
*   Users can test regular queries, blacklist commands (e.g., "don't talk about sports"), and see how the system combines information or respects blacklists.
*   Type 'quit' to exit the loop.

*(Optional debug print statements within the ChatManager might show intermediate steps like context fetching and blacklist updates.)*

In [ ]:
# chat = ChatManager(llm, tavily_tool, retriever, rag_chain)
# while True:
#     user_input = input("You: ")
#     if not user_input:
#         break
#     print("Bot: ", chat.handle_user_message(user_input))


In [ ]:
# You could try checking out the memory contents at any point.
#print([msg for msg in chat.memory.chat_memory.messages if isinstance(msg, HumanMessage)])


##### **11. Medical AI Agent**

**11.1 RAG tool**

In [24]:
from langchain.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import Tool

# Define the system prompt that will guide the chatbot
system_prompt = (
    "You are a medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise. If the question is not clear, ask follow-up questions."
    "\n\n"
    "{context}"
)

# Create a chat prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# Initialize the LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

# Create the RAG chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)



# Define a wrapper callable that matches the expected input signature for a Tool
def rag_context_tool(user_input):
    try:
      rag_response = rag_chain.invoke({"input": user_input})
      return rag_response.get("answer", "Sorry, no RAG answer found.")
    except Exception as e:
      return f"Error retrieving RAG context: {e}"

rag_context_tool = Tool(
    name="RAG Context Retriever",
    func=rag_context_tool,
    description="Retrieves relevant RAG context given user input and chat history."
)

**11.2 Memory setup**

In [25]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

**11.2 AI Agent with RAG and Tavily tools**

In [26]:
from langchain.agents.react.agent import create_react_agent
from langchain.agents.agent import AgentExecutor
from langchain.llms import OpenAI
from langchain.tools import Tool
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

# Your tools
tools = [tavily_tool, rag_context_tool]

# Your custom prompt including all required LangChain agent variables
# https://python.langchain.com/api_reference/langchain/agents/langchain.agents.react.agent.create_react_agent.html
template = """
You are a helpful medical assistant. Get an answer based only on the internal (RAG) and web (Tavily) context.
Do not answer it on your own.

If the question is non-medical, respond:
Final Answer: I'm not allowed to discuss it.

Tools:
{tools}

Use the following format:

Question: {input}

Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action


Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
"""

prompt = PromptTemplate(
    input_variables=["input", "tools", "tool_names", "agent_scratchpad"],
    template=template
)


agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, memory=memory, verbose=True, output_key="output")


**11.4 Testing medical AI agent**

In [ ]:
response = agent_executor.invoke({"input": "hi"})
print(response)

In [27]:
response = agent_executor.invoke({"input": "what is capital of India"})
print(response)



> Entering new AgentExecutor chain...
Final Answer: I'm not allowed to discuss it.

> Finished chain.
{'input': 'what is capital of India', 'chat_history': [HumanMessage(content='what is capital of India', additional_kwargs={}, response_metadata={}), AIMessage(content="I'm not allowed to discuss it.", additional_kwargs={}, response_metadata={})], 'output': "I'm not allowed to discuss it."}


In [28]:
response = agent_executor.invoke({"input": "what is optical nerve damage treatment"})
print(response)



> Entering new AgentExecutor chain...
Action: RAG Context Retriever
Action Input: optical nerve damage treatmentOptical nerve damage is generally considered irreversible, and doctors may not be able to do much to restore lost vision. However, eye drops like Brimonidine may theoretically help in neuroprotection and preserve any remaining optic nerve function. The management plan often focuses on preserving what function is left rather than reversing the damage.I now know the final answer
Final Answer: Optical nerve damage is generally considered irreversible, and doctors may not be able to do much to restore lost vision. However, eye drops like Brimonidine may theoretically help in neuroprotection and preserve any remaining optic nerve function. The management plan often focuses on preserving what function is left rather than reversing the damage.

> Finished chain.
{'input': 'what is optical nerve damage treatment', 'chat_history': [HumanMessage(content='what is capital of India', ad

In [29]:
response = agent_executor.invoke({"input": "what is osteoporosis"})
print(response)



> Entering new AgentExecutor chain...
Action: RAG Context Retriever
Action Input: osteoporosisThe provided context mentions osteopenia, a condition characterized by lower than normal bone mineral density. For this condition, the doctor advises continuing calcium tablets and consulting an orthopedic surgeon for medications to help build up calcium. The context does not provide specific information about osteoporosis.Action: tavily_search_results_json
Action Input: what is osteoporosis[{'title': 'About Osteoporosis', 'url': 'https://www.osteoporosis.foundation/patients/about-osteoporosis', 'content': 'Skip to main content\n\n## What is osteoporosis?\n\nOsteoporosis is a disease which makes bones weak and fragile. This greatly increases the risk of breaking a bone even after a minor fall or bump. The disease has no obvious symptoms, so many people don’t know they have osteoporosis until they suffer a fracture. Fractures can be life-altering, causing pain, disability and loss of independ